# TIỀN XỬ LÝ DỮ LIỆU & CHUẨN HÓA (PREPROCESSING & SCALING)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

# Đọc dữ liệu Ames Housin

In [2]:
train_df = pd.read_csv('data/train.csv')

# Xóa Outliers cực đoan theo kết quả Lab 03 (Id 524 và 1299)

In [3]:
train_df = train_df.drop(train_df[(train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 300000)].index)

# Tách Biến mục tiêu - Target và Đặc trưng - Features

In [4]:
X = train_df.drop(columns=['Id', 'SalePrice'])
y = np.log1p(train_df['SalePrice']) # Biến đổi Log1p cho SalePrice

# Phân loại biến Số - Numerical và biến Chữ - Categorical

In [5]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_9936\2340483407.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns


# Xử lý Missing Values

## Điền missing biến số bằng Median

In [6]:
imputer_num = SimpleImputer(strategy='median')
X[num_cols] = imputer_num.fit_transform(X[num_cols])


## Điền missing biến phân loại bằng 'None'

In [7]:
X[cat_cols] = X[cat_cols].fillna('None')

# Categorical Encoding - Chuyển chữ thành số

In [8]:
X_encoded = pd.get_dummies(X, drop_first=True)

# Chia tập Train / Test (80% / 20%)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)


# Chuẩn hóa dữ liệu 

In [10]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_encoded.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_encoded.columns)

print(f"HOÀN THÀNH BƯỚC TIỀN XỬ LÝ")
print(f"Kích thước X_train sau mã hóa & chuẩn hóa: {X_train_scaled.shape}")

HOÀN THÀNH BƯỚC TIỀN XỬ LÝ
Kích thước X_train sau mã hóa & chuẩn hóa: (1166, 259)


# FILTER METHODS & WRAPPER METHOD (RFE)

In [11]:
from sklearn.feature_selection import mutual_info_regression, RFE
from sklearn.ensemble import RandomForestRegressor

# FILTER METHOD - CORRELATION
## Tính tương quan của từng thuộc tính với biến mục tiêu y


In [12]:
corr_with_y = X_train_scaled.apply(lambda col: col.corr(y_train)).abs()
selected_corr = corr_with_y[corr_with_y > 0.3].index.tolist()

c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


# FILTER METHOD - MUTUAL INFORMATION


In [13]:
mi_scores = mutual_info_regression(X_train_scaled, y_train, random_state=42)
mi_series = pd.Series(mi_scores, index=X_train_scaled.columns)
selected_mi = mi_series.nlargest(20).index.tolist()

# WRAPPER METHOD - RFE (RECURSIVE FEATURE ELIMINATION)

In [14]:
rfe_selector = RFE(
    estimator=RandomForestRegressor(n_estimators=50, random_state=42),
    n_features_to_select=20
)
rfe_selector.fit(X_train_scaled, y_train)
selected_rfe = X_train_scaled.columns[rfe_selector.support_].tolist()

print(f"Filter (Correlation > 0.3): Chọn {len(selected_corr)} đặc trưng.")
print(f"Filter (Mutual Info Top 20): Chọn {len(selected_mi)} đặc trưng.")
print(f"Wrapper (RFE Top 20): Chọn {len(selected_rfe)} đặc trưng.")


Filter (Correlation > 0.3): Chọn 0 đặc trưng.
Filter (Mutual Info Top 20): Chọn 20 đặc trưng.
Wrapper (RFE Top 20): Chọn 20 đặc trưng.
